In [ ]:
import os
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# matplotlib settings
plt.rc("text", usetex=True)
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Computer Modern Roman"]
plt.rcParams["font.size"] = 12
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["text.latex.preamble"] = r"\usepackage{xfrac}\usepackage{siunitx}"

In [ ]:
def sq_exp_kernel(x1, x2, *, l, w):
    if not isinstance(x1, np.ndarray):
        x1 = np.array([[x1]])
    if not isinstance(x2, np.ndarray):
        x2 = np.array([[x2]])
    if len(x1.shape) == 1:
        x1 = x1.reshape((-1, 1))
    if len(x2.shape) == 1:
        x2 = x2.reshape((-1, 1))

    X1 = np.tile(x1, (1, x2.shape[0]))
    X2 = np.tile(x2.T, (x1.shape[0], 1))
    return w * np.exp(-((X1 - X2) ** 2) / l**2)


def periodic_kernel(x1, x2, *, l, w, p):
    if not isinstance(x1, np.ndarray):
        x1 = np.array([[x1]])
    if not isinstance(x2, np.ndarray):
        x2 = np.array([[x2]])
    if len(x1.shape) == 1:
        x1 = x1.reshape((-1, 1))
    if len(x2.shape) == 1:
        x2 = x2.reshape((-1, 1))

    X1 = np.tile(x1, (1, x2.shape[0]))
    X2 = np.tile(x2.T, (x1.shape[0], 1))

    return w * np.exp(-2 * np.sin(np.abs(X1 - X2) * np.pi / p) ** 2 / l**2)

In [ ]:
l = 2.0
w = 1.0
p = 1.0
n_dense = 101
n_sample = 20

x_dense = np.linspace(0, 1, n_dense)
x_train = np.array([[0.3]])
y_train = np.array([2.5])
x_test = np.array([0.5])
obsnoise = 1e-4 * np.identity(1)
pdnoise = 1e-8 * np.identity(n_dense)

In [ ]:
mu_prior = np.zeros_like(x_dense)
Sigma_prior = periodic_kernel(x_dense, x_dense, l=l, w=w, p=p)
std_prior = np.sqrt(np.diagonal(Sigma_prior))

Sigma_cross = periodic_kernel(x_dense, x_train, l=l, w=w, p=p)
Sigma_obs = periodic_kernel(x_train, x_train, l=l, w=w, p=p)
Gram_inv = np.linalg.inv(Sigma_obs + obsnoise)

mu_post = Sigma_cross @ Gram_inv @ y_train
Sigma_post = Sigma_prior - Sigma_cross @ Gram_inv @ Sigma_cross.T
std_post = np.sqrt(np.diagonal(Sigma_post))

L_prior = np.linalg.cholesky(Sigma_prior + pdnoise)
L_post = np.linalg.cholesky(Sigma_post + pdnoise)

rng = np.random.default_rng(0)
w = rng.standard_normal((n_dense, n_sample))
sample_prior = np.tile(mu_prior, (n_sample, 1)).T + L_prior @ w
sample_post = np.tile(mu_post, (n_sample, 1)).T + L_post @ w

idx_test = np.where(np.isclose(x_dense, x_test))[0][0]
mu_prior_test = mu_prior[idx_test]
sigma_prior_test = np.sqrt(Sigma_prior[idx_test, idx_test])
samples_prior_test = sample_prior[idx_test, :]
mu_post_test = mu_post[idx_test]
sigma_post_test = np.sqrt(Sigma_post[idx_test, idx_test])
samples_post_test = sample_post[idx_test, :]

y_dense = np.linspace(-4, 4, 100)
p_prior = norm.pdf(y_dense, loc=mu_prior_test, scale=sigma_prior_test)
p_post = norm.pdf(y_dense, loc=mu_post_test, scale=sigma_post_test)

In [ ]:
color_prior = "0.5"
color_post = sns.cm.rocket(0.5)
width_sample = 0.4
width_mean = 1.0
alpha_cov = 0.2
alpha_scatter = 0.7

fig, (ax1, ax2) = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(7.2, 3.6),
    sharey=True,
    tight_layout=True,
    gridspec_kw={"width_ratios": [3, 1]},
)

ax1.plot(x_dense, mu_prior, color=color_prior, linewidth=width_mean, label=r"$u$")
ax1.fill_between(
    x_dense,
    mu_prior - 2 * std_prior,
    mu_prior + 2 * std_prior,
    color=color_prior,
    alpha=alpha_cov,
)
ax1.plot(x_dense, sample_prior, color=color_prior, linewidth=width_sample)
ax1.plot(
    x_dense, mu_post, color=color_post, linewidth=width_mean, label=r"$u|\mathcal{L}$"
)
ax1.fill_between(
    x_dense,
    mu_post - 2 * std_post,
    mu_post + 2 * std_post,
    color=color_post,
    alpha=alpha_cov,
)
ax1.plot(x_dense, sample_post, color=color_post, linewidth=width_sample)
ax1.scatter(x_train, y_train, color="k", marker="o", zorder=3)
ax1.text(x_train, y_train + 0.3, r"$\mathcal{L}$")
ax1.axvline(x_test, color="k", linestyle="--")
ax1.text(x_test - 0.04, -3.5, r"$\tilde{\mathcal{L}}$")
ax1.legend(loc="lower right")
ax1.set_xlim((0.0, 1.0))
ax1.set_ylim((-3.0, 4.0))
ax1.set_xticks([0.0, 0.5, 1.0])
ax1.set_yticks([-4.0, -2.0, 0.0, 2.0, 4.0])
ax1.set_xlabel("$x$")
ax1.set_ylabel("$y$")

ax2.plot(
    p_prior,
    y_dense,
    color=color_prior,
    linewidth=width_mean,
    label=r"$\tilde{\mathcal{P}}u$",
)
ax2.scatter(
    np.zeros(n_sample),
    samples_prior_test,
    color=color_prior,
    marker=".",
    alpha=alpha_scatter,
    zorder=3,
    ec="none",
)
ax2.plot(
    p_post,
    y_dense,
    color=color_post,
    linewidth=width_mean,
    label=r"$\tilde{\mathcal{P}}u|\mathcal{L}$",
)
ax2.scatter(
    np.zeros(n_sample),
    samples_post_test,
    color=color_post,
    marker=".",
    alpha=alpha_scatter,
    zorder=3,
    ec="none",
)
ax2.set_xticks([])
ax2.set_xlabel("$p(y)$")
ax2.legend(loc="lower right")

# fname = "1d-analogy.pdf"
# fname = os.path.join("plots", fname)
# os.makedirs(os.path.dirname(fname), exist_ok=True)
# plt.savefig(fname, bbox_inches="tight")

plt.show()